# Lab 3: Quantum algorithms: hidden strings and search

**Audience:** comfortable with Python and basic linear algebra; no prior quantum computing required for Lab 1.

## Learning objectives

- Explain an oracle and distinguish oracle queries from total computational cost.
- Use phase kickback to implement Bernstein–Vazirani with explicit gates.
- Build a two-qubit Grover oracle and diffuser.
- Explain why too many Grover iterations can reduce success.

## How to use this notebook

Open [Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**, and upload this file. Connect to a standard Python CPU runtime. Run cells in order with Shift+Enter. No IBM account, token, GPU, or paid service is needed. Internet access is needed for package installation only. If changing an already-used package version, restart the session after installation and rerun from the imports cell. These notebooks target Python 3.10–3.13.

For each experiment: **explain → predict → run → interpret → modify**. Write predictions and observations in a new text cell or your notes. Exercise starter cells are working examples, not finished answers to every requested change. Hints and solutions are collapsed below the prompts. Keep optional extensions until after the core lab.

## Agenda

| Activity |
|---|
| Recap, oracles, query complexity; Exercise 1 |
| Bernstein–Vazirani and phase kickback |
| Hidden-string experiments; Exercise 2 |
| Break and checkpoint |
| Grover oracle, diffusion, amplitudes; Exercise 3 |
| Marked states and iteration counts; Exercise 4 |
| Exit questions and limitations |

## Prerequisite recap

H creates plus/minus states; a relative phase can affect later measurement through interference. CNOT acts linearly and can correlate qubits. Qiskit labels computational-basis strings with the highest qubit index on the left. All imports and helpers are repeated here; no earlier kernel state is required.

**Working together:** compare predictions with a partner before running code. If you finish early, try an optional extension.


## Setup

The first cell installs a fixed teaching environment. Installation may take a few minutes. The helper functions below build explicit classical measurement registers and use 2,048 shots with seed 42. A shot means one fresh preparation and measurement of the circuit. Fixed seeds make classroom results repeatable; they do not remove sampling uncertainty.

`exact` inspects a simulated state without measurement. `sample` returns finite-shot counts. `compare` plots both. Access to all amplitudes is a simulator convenience: a physical quantum computer does not hand you its state vector.


In [ ]:
%pip install -q qiskit==2.2.3 numpy==2.2.6 matplotlib==3.10.3 pylatexenc==2.10 sympy==1.14.0


In [ ]:
import sys
from importlib.metadata import version
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, DensityMatrix
from qiskit.primitives import StatevectorSampler

SEED, SHOTS = 42, 2048
print('Python:', sys.version.split()[0])
for package in ['qiskit', 'numpy', 'matplotlib', 'pylatexenc', 'sympy']:
    print(f'{package}: {version(package)}')

def exact(circuit):
    # Supply a circuit WITHOUT measurements. This is simulator-only inspection.
    return Statevector.from_instruction(circuit).probabilities_dict()

def sample(circuit, qubits=None):
    # Copy so that the original circuit stays available for state inspection.
    qubits = list(range(circuit.num_qubits)) if qubits is None else list(qubits)
    measured = circuit.copy()
    readout = ClassicalRegister(len(qubits), 'readout')
    measured.add_register(readout)
    measured.measure(qubits, readout)
    sampler = StatevectorSampler(seed=SEED)
    return sampler.run([measured], shots=SHOTS).result()[0].data.readout.get_counts()

def compare(circuit, title):
    probabilities, counts = exact(circuit), sample(circuit)
    labels = [format(i, f'0{circuit.num_qubits}b') for i in range(2**circuit.num_qubits)]
    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.bar(x - .18, [probabilities.get(k, 0) for k in labels], .36, label='Exact probability')
    ax.bar(x + .18, [counts.get(k, 0) / SHOTS for k in labels], .36, label=f'Sampled frequency ({SHOTS} shots)')
    ax.set(xticks=x, xticklabels=labels, ylim=(0, 1.12), ylabel='Probability / frequency', xlabel='Measured bit string', title=title)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
    print('Counts:', counts)
    return counts

def show(circuit):
    display(circuit.draw('mpl', fold=70))
    plt.show()


## What is an oracle?

An **oracle** is a specified interface to a function. A reversible bit oracle implements `U_f|x⟩|y⟩=|x⟩|y⊕ f(x)⟩`, where `⊕` is XOR. It can be applied to superpositions, but measurement does not reveal every function value.

**Query complexity** counts calls to this interface, treating each call as one unit. It leaves out gate construction, state preparation, communication, and measurement overhead. Our small oracles are built from known answers so that we can study algorithm behavior. This is an educational black-box model, not a claim that discovering an answer and building an oracle is free.


### Exercise 1 — Read a reversible oracle

For f(x)=x on one input bit, a CNOT implements the oracle. Predict outputs for all four pairs (x,y), then run. Explain why overwriting y with f(x), even while retaining x, loses the original y and is not reversible.

Record your prediction before editing the baseline cell below. Run it, explain the result, then make the requested change.

<details><summary>Hint</summary>

The target becomes y XOR x, and the control is unchanged. Applying CNOT twice restores the input.

</details>

<details><summary>Worked solution — open after attempting</summary>

For (x,y)=00,01,10,11 the output pairs are 00,01,11,10. The printed Qiskit strings are yx, so inspect that ordering carefully. Overwriting y maps (x,0) and (x,1) to the same (x,f(x)), so the original y cannot be recovered. XOR preserves it: applying the same oracle twice restores both inputs.

</details>


In [ ]:
for x in [0, 1]:
    for y in [0, 1]:
        oracle_demo = QuantumCircuit(2)
        if x: oracle_demo.x(0)
        if y: oracle_demo.x(1)
        oracle_demo.cx(0, 1)
        print(f'Input (x,y)=({x},{y}); output string yx:', sample(oracle_demo))


## Bernstein–Vazirani

An unknown n-bit string `s` defines `f_s(x)=s· x (mod 2)`, the parity of positions where both s and x are 1. A deterministic classical algorithm needs n queries to recover s exactly: query each unit vector. One quantum oracle query recovers all n bits in the ideal model.

### Phase kickback

Prepare an auxiliary qubit in `|-⟩` using X then H. Since `X|-⟩=-|-⟩`, the oracle acts as

```text
U_f|x⟩|-⟩=(-1)^(f(x))|x⟩|-⟩.
```

The auxiliary state factors out, while the input amplitudes gain x-dependent signs. This is **phase kickback**. The oracle has not copied the auxiliary qubit's measurement result into the input.

Start all input qubits in plus states. Each s bit equal to 1 makes its corresponding input effectively a minus state after the parity oracle. Final H gates turn plus to zero and minus to one. Thus measuring inputs recovers s.

We define s in displayed order: `1010` means s3=1, s2=0, s1=1, s0=0. The auxiliary qubit is q[n] and is not measured.


In [ ]:
def parity_oracle(secret):
    if not secret or set(secret) - {'0', '1'}:
        raise ValueError('Secret must be a nonempty binary string.')
    n = len(secret)
    oracle = QuantumCircuit(n + 1, name='Uf')
    for q, bit in enumerate(reversed(secret)):
        if bit == '1': oracle.cx(q, n)
    return oracle

def bernstein_vazirani(secret):
    n = len(secret)
    qc = QuantumCircuit(n + 1)
    qc.x(n)
    qc.h(n)                  # auxiliary is minus
    qc.h(range(n))           # inputs are uniform
    qc.barrier(label='one oracle query')
    qc.compose(parity_oracle(secret), inplace=True)
    qc.barrier()
    qc.h(range(n))           # interference reveals the secret
    return qc

secret = '1010'
bv = bernstein_vazirani(secret)
show(bv)
print('Recovered input string:', sample(bv, range(len(secret))))
print('Exact input probabilities:', Statevector.from_instruction(bv).probabilities(qargs=list(range(len(secret)))))


### Trace the classical and quantum data

**Predict:** What happens for an all-zero secret? What if the auxiliary starts in plus rather than minus?

**Interpret:** the oracle contains one CNOT for each 1 in the secret. Counting the entire oracle as one query does not make all these gates cost one operation. The advantage here is n versus one calls to the specified function interface.


### Exercise 2 — Recover new secrets

Test 0000, 0001, 1101, and your own five-bit secret. Implement the classical unit-vector queries and compare the query counts. Then change the auxiliary preparation from minus to plus and predict the result.

Record your prediction before editing the baseline cell below. Run it, explain the result, then make the requested change.

<details><summary>Hint</summary>

Use `int(secret, 2)` and bitwise AND to compute a parity, or take a dot product modulo 2. Plus is an eigenstate of X with eigenvalue +1.

</details>

<details><summary>Worked solution — open after attempting</summary>

Every ideal quantum run recovers its secret. A classical exact strategy queries each of n basis strings, obtaining the corresponding secret bit. With auxiliary plus, no phase is kicked back, so the final input is all zeros regardless of s.

</details>


In [ ]:
def classical_unit_queries(secret):
    n = len(secret)
    answers = []
    for q in range(n):
        x = 1 << q
        answers.append((int(secret, 2) & x).bit_count() % 2)
    return ''.join(map(str, reversed(answers))), n

for secret in ['0000', '0001', '1101', '10101']:
    qc = bernstein_vazirani(secret)
    print(secret, 'quantum:', sample(qc, range(len(secret))),
          'classical (answer, queries):', classical_unit_queries(secret))


## Break and checkpoint

Explain how a sign change can carry information without changing the intermediate Z probabilities. Distinguish an oracle query from a physical gate. Take a break before the search experiment.


## Grover search among four candidates

Let exactly one of `00`, `01`, `10`, `11` be marked. Begin with equal amplitudes `1/2`. A **phase oracle** negates only the marked amplitude. This sign change alone leaves measurement probabilities unchanged.

The diffuser `D=2|u⟩⟨ u|-I`, where `|u⟩` is the uniform state, maps each amplitude `aₓ` to `2mean(a)-aₓ` (inversion about the mean). After the oracle, the amplitudes are three copies of 1/2 and one of −1/2; their mean is 1/4. One diffusion step gives marked amplitude 1 and all others 0.

### Build a two-qubit phase oracle

CZ negates only `11`. To mark any target, apply X to positions where the target has zero, apply CZ, then undo those X gates. Target strings follow q1q0 ordering.

### Build the diffuser

H on both → X on both → CZ → X on both → H on both implements `-D`; an extra global phase π makes it exactly D for easier amplitude tracing. Global phase does not change measurements.

**Predict:** At which stage do probabilities change? Why is the phase oracle alone insufficient?


In [ ]:
def mark_target(circuit, target):
    if len(target) != 2 or set(target) - {'0', '1'}:
        raise ValueError('Use a two-bit target.')
    zeros = [q for q, bit in enumerate(reversed(target)) if bit == '0']
    for q in zeros: circuit.x(q)
    circuit.cz(0, 1)
    for q in zeros: circuit.x(q)

def diffuse(circuit):
    circuit.h([0, 1])
    circuit.x([0, 1])
    circuit.cz(0, 1)
    circuit.x([0, 1])
    circuit.h([0, 1])
    circuit.global_phase += np.pi

def grover(target, iterations=1):
    qc = QuantumCircuit(2)
    qc.h([0, 1])
    for _ in range(iterations):
        mark_target(qc, target)
        diffuse(qc)
    return qc

target = '10'
uniform = QuantumCircuit(2)
uniform.h([0, 1])
marked = uniform.copy()
mark_target(marked, target)
amplified = marked.copy()
diffuse(amplified)
for label, qc in [('Uniform', uniform), ('After oracle', marked), ('After diffusion', amplified)]:
    print(label, 'amplitudes in 00,01,10,11 order:', np.round(Statevector.from_instruction(qc).data, 3))
    compare(qc, label)
show(amplified)


### Exercise 3 — Construct another oracle

Change the target to 01. Predict which qubit needs X around CZ. Inspect the state after the oracle and verify that only the target amplitude changes sign. Then finish one iteration.

Record your prediction before editing the baseline cell below. Run it, explain the result, then make the requested change.

<details><summary>Hint</summary>

String 01 has q1=0, q0=1; conjugate CZ with X on q1.

</details>

<details><summary>Worked solution — open after attempting</summary>

Apply X(1), CZ(0,1), X(1). The oracle amplitudes are `[.5,-.5,.5,.5]`. Diffusion produces `[0,1,0,0]` up to numerical roundoff, so measurement returns 01.

</details>


In [ ]:
target = '11'  # Change to 01
oracle_check = QuantumCircuit(2)
oracle_check.h([0, 1])
mark_target(oracle_check, target)
print('Oracle amplitudes:', np.round(Statevector.from_instruction(oracle_check).data, 3))
diffuse(oracle_check)
compare(oracle_check, f'Grover finds {target}')


## More iterations are not always better

With one marked item among N candidates, the ideal success probability after r iterations is `sin²((2r+1)θ)`, with `sinθ=1/√N`. Here N=4 and `θ=π/6`. One iteration succeeds with probability 1, but two iterations return to probability 1/4: repeated steps rotate past the desired state.

For large N and one solution, Grover uses order `√N` oracle queries for high success, versus order N classical queries. Our four-item example illustrates the mechanism; it does not demonstrate a practical speedup on this classical simulator.


### Exercise 4 — Search and overshoot

Check all four targets after one iteration. Then sweep iterations 0 through 5 for a fixed target. Predict the curve before running, compare it with the formula, and explain why simply increasing the iterations is a poor strategy.

Record your prediction before editing the baseline cell below. Run it, explain the result, then make the requested change.

<details><summary>Hint</summary>

For N=4, use theta=pi/6. Exact target probabilities can be read with `exact(qc).get(target, 0)`.

</details>

<details><summary>Worked solution — open after attempting</summary>

All four targets succeed with probability 1 after one iteration. For r=0…5, success is `[.25,1,.25,.25,1,.25]`. Amplitude amplification is a rotation rather than a monotonic improvement.

</details>


In [ ]:
for target in ['00', '01', '10', '11']:
    qc = grover(target)
    print('Target:', target, 'exact success:', exact(qc).get(target, 0), 'counts:', sample(qc))

target = '10'
iterations = np.arange(6)
success = [exact(grover(target, int(r))).get(target, 0) for r in iterations]
predicted = np.sin((2 * iterations + 1) * np.pi/6)**2
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(iterations, success, 'o-', label='Exact circuit probability')
ax.plot(iterations, predicted, 'x--', label='Analytical formula')
ax.set(xlabel='Grover iterations / oracle queries', ylabel='Target probability', ylim=(0, 1.1), xticks=iterations)
ax.legend()
fig.tight_layout()
plt.show()
assert np.allclose(success, predicted)


## Exit ticket

1. What is the oracle interface in Bernstein–Vazirani, and what does one quantum query recover?
2. How does phase kickback differ from reading out every function value?
3. Why does Grover need both a phase oracle and diffusion?
4. Why do these demonstrations not establish a practical runtime advantage?

**Optional extension:** remove the X preparation of the BV auxiliary so it starts in plus, verify all-zero input output, and derive why. Alternatively, write the 4×4 diffuser matrix `2*np.ones((4,4))/4 - np.eye(4)` and compare its action with the gate implementation.

## References

- [IBM: quantum algorithms course](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms)
- [IBM: Grover tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/grovers-algorithm)
- [IBM: Qiskit bit ordering](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering)
- [Qiskit Statevector](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector)
